In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
here = Path.cwd()
datadir_csv = here.parent/"Data_CSVs/"
#print(datadir)
datadir = here.parent/"SimulationScripts/tempXYZ/"
cmap = plt.cm.get_cmap('BuPu')
clrs=["#c7e9b4","#7fcdbb","#41b6c4","#1d91c0","#225ea8","#0c2c84","#c7e9b4","#7fcdbb","#41b6c4","#1d91c0","#225ea8","#0c2c84"]
clrs2=["#b59837","#fc8d59","#d7301f","#6f0606","#4575b4","#313695","#313695","#4575b4","#91bfdb","#313695","#4575b4","#313695","#fef0d9","#fdcc8a","#fc8d59","#d7301f","#4575b4","#313695","#313695","#4575b4","#91bfdb","#313695","#4575b4","#313695"]
clrs3=["#313695","#623CA2","#B546CE"]

# simulates the following equation of motion:  
$\gamma\dot{\bold{x}} = \bold{F}_{\mathrm{env}} + \bold{f}^{A}$  
$\dot{\bold{f}}^{A} = \frac{-1}{\tau_{\mathrm{mem}}}\bold{f}^{A} + \eta$  

My preferred output is a .xyz file which can be dropped directly into the app 'Ovito' (download here: https://www.ovito.org/) for checking the dynamics

Simulation definitions:

In [5]:
def parse_lammps_xyz(filename):
    data = []
    with open(filename, 'r') as file:
        while True:
            line = file.readline()
            if not line:
                break  # EOF
            
            if "ITEM: TIMESTEP" in line:
                timestep = int(file.readline().strip())
                file.readline()  # Skip "ITEM: NUMBER OF ATOMS"
                num_atoms = int(file.readline().strip())
                file.readline()  # Skip "ITEM: BOX BOUNDS ..."
                file.readline()
                file.readline()
                file.readline()
                file.readline()  # Skip "ITEM: ATOMS id type x y z"
                
                for _ in range(num_atoms):
                    atom_line = file.readline()
                    if not atom_line:
                        break  # Unexpected EOF
                    parts = atom_line.strip().split()
                    atom_id, atom_type = int(parts[0]), int(parts[1])
                    x, y, z, sumForce = map(float, parts[2:6])
                    data.append({
                        "timestep": timestep,
                        "id": atom_id,
                        "type": atom_type,
                        "x": x,
                        "y": y,
                        "z": z,
                        "sumForce": sumForce
                    })

    df = pd.DataFrame(data)
    return df

# Unwrapping the trajectories
# The function unwrap_trajectories takes a DataFrame with atom trajectories and unwraps them based on periodic boundary conditions.

def unwrap_trajectories(df, box_length_x, box_length_y):
    # Sort the DataFrame by atom ID and timestep
    df = df.sort_values(by=["id", "timestep"]).copy()

    # Group by atom ID
    unwrapped_x = []
    unwrapped_y = []

    for atom_id, group in df.groupby("id"):
        group = group.sort_values(by="timestep").copy()
        x = group["x"].values
        y = group["y"].values
        timesteps = group["timestep"].values
        if x[0]<0:
            unwrappedX = [x[0]+box_length_x]
        elif x[0]>box_length_x:
            unwrappedX = [x[0]-box_length_x]
        else:
            unwrappedX = [x[0]]
        if y[0]<0:
            unwrappedY = [y[0]+box_length_y]
        elif y[0]>box_length_y:
            unwrappedY = [y[0]-box_length_y]
        else:
            unwrappedY = [y[0]]
        
        shiftX = 0
        shiftY = 0

        for i in range(1, len(x)):
            dx = x[i] - x[i - 1]
            dy = y[i] - y[i - 1]
            if dx > box_length_x / 2:
                shiftX -= box_length_x
            if dy > box_length_y / 2:
                shiftY -= box_length_y
            elif dx < -box_length_x / 2:
                shiftX += box_length_x
            elif dy < -box_length_y / 2:
                shiftY += box_length_y
            unwrappedX.append(x[i] + shiftX)
            unwrappedY.append(y[i] + shiftY)

        unwrapped_x.extend(unwrappedX)
        unwrapped_y.extend(unwrappedY)

    df["x_unwrapped"] = unwrapped_x
    df["y_unwrapped"] = unwrapped_y
    return df

# Function to write positions to .xyz file
def write_xyz(f, positions, step,summing_forces,box_length):
    #with open(f, 'a') as f:
    f.write("ITEM: TIMESTEP\n")
    f.write(f"{step}\n")
    f.write("ITEM: NUMBER OF ATOMS\n")
    f.write(f"{len(positions)}\n")
    # Write box dimensions (optional)
    f.write("ITEM: BOX BOUNDS pp pp pp\n")
    f.write(f"{-box_length/2} {box_length/2}\n")
    f.write(f"{-box_length/2} {box_length/2}\n")
    f.write(f"0.0 0.1\n")
    f.write("ITEM: ATOMS id type x y z sumForce\n")
    i=0
    for pos in positions:
        i+=1
        f.write(f"{i} 1 {pos[0]} {pos[1]} {0} {summing_forces[i-1]}\n") 
        #f.write(f"1 {pos[0]} {pos[1]} 0.0\n")  # Assuming particles are Argon atoms
        
# Function to compute harmonic trap force (constant gradient: F = -kappa * displacement)
def harmonic_force(positions, kappa, center): 

    displacement = positions - center
    return -kappa * displacement

    




Simulation itself:

In [9]:
class VerletNeighborList:
    """Verlet neighbour list with skin-distance rebuild criterion."""

    def __init__(self, cutoff, skin, box_size):
        self.cutoff = float(cutoff)
        self.skin = float(skin)
        self.cutoff_with_skin = self.cutoff + self.skin
        self.rebuild_threshold = 0.5 * self.skin
        self.box_size = float(box_size)
        self.pairs = None
        self.reference_positions = None

    def _minimum_image(self, dr):
        return dr - self.box_size * np.round(dr / self.box_size)

    def needs_rebuild(self, positions):
        if self.pairs is None or self.reference_positions is None:
            return True
        displacement = self._minimum_image(positions - self.reference_positions)
        max_disp = np.max(np.linalg.norm(displacement, axis=1))
        return max_disp > self.rebuild_threshold

    def build(self, positions):
        num_particles = len(positions)
        pairs = []
        for i in range(num_particles):
            for j in range(i + 1, num_particles):
                rij = self._minimum_image(positions[i] - positions[j])
                r = np.linalg.norm(rij)
                if r < self.cutoff_with_skin:
                    pairs.append((i, j))
        self.pairs = pairs
        self.reference_positions = positions.copy()
        return pairs

    def get_pairs(self, positions):
        if self.needs_rebuild(positions):
            return self.build(positions)
        return self.pairs


def compute_soft_lj_forces(positions, box_size, epsilon, sigma, rcut, lambda_param, neighbor_pairs=None):
    n, alpha_lj = 1, 0.5
    num_particles = len(positions)
    forces = np.zeros_like(positions)
    rcut2 = rcut ** 2
    lam_pow_n = lambda_param ** n
    const_factor = 4 * epsilon * lam_pow_n
    last_force = 0.0

    if neighbor_pairs is None:
        pair_iterator = ((i, j) for i in range(num_particles) for j in range(i + 1, num_particles))
    else:
        pair_iterator = neighbor_pairs

    for i, j in pair_iterator:
        rij = positions[i] - positions[j]
        rij -= box_size * np.round(rij / box_size)
        r2 = np.dot(rij, rij)

        if r2 >= rcut2 or r2 <= 1e-12:
            continue

        r = np.sqrt(r2)
        r6 = (r / sigma) ** 6
        A = alpha_lj * (1 - lambda_param) ** 2 + r6
        dAdr = 6 * (r / sigma) ** 5 / sigma
        dEdr = const_factor * (-2 / A**3 + 1 / A**2) * dAdr
        force_vec = -dEdr * (rij / r)

        forces[i] += force_vec
        forces[j] -= force_vec
        last_force = -dEdr

    return forces, last_force

    



def place_particles_in_box_crystal(num_particles, box_size, delta):
    """
    Place particles on a 2D square crystal lattice inside a square box,
    keeping all particles at least `delta` away from the walls.

    Box: [0, L] x [0, L]

    Args:
        num_particles: int
        box_size: float (L)
        delta: float (minimum distance from walls)

    Returns:
        positions: (N, 2) array
    """
    # number of lattice points per side
    n_side = int(np.ceil(np.sqrt(num_particles)))

    # usable length after wall exclusion
    L_eff = box_size - 2.0 * delta
    if L_eff <= 0:
        raise ValueError("delta is too large for the box")

    spacing = L_eff / n_side

    if spacing <= 0:
        raise ValueError("Particles do not fit with the given delta")

    positions = np.zeros((num_particles, 2))

    idx = 0
    for i in range(n_side):
        for j in range(n_side):
            if idx >= num_particles:
                break
            positions[idx] = np.array([
                delta + (i + 0.5) * spacing-box_size/2,
                delta + (j + 0.5) * spacing-box_size/2
            ])
            idx += 1

    return positions


def OD_MD_SimulationLJPairs(num_particles, box_size, kappa, temperature, gamma, dt,
                     num_steps0, equilibration_steps, frame, use_harmonic_trap,
                     name, center, epsilon, sigma, rcut, lambda_param,
                     use_neighbor_list=False, neighbor_skin=1.0,
                     collect_trajectory=False, random_seed=None, walls = False,tau=100,
                     use_arbitrary_trap=False, kappa_y=None, A_x=1.0, L_x=1.0):
    rng = np.random.default_rng(random_seed)
    #positions = rng.uniform(-box_size / 2, box_size / 2, size=(num_particles, 2))
    positions = place_particles_in_box_crystal(num_particles, box_size,1)
    summing_forces = np.zeros(num_particles)
    f_active = np.zeros((num_particles,2))

    num_steps = num_steps0 + equilibration_steps
    min_ljForce = 0.0

    neighbor_list = None
    if use_neighbor_list:
        neighbor_list = VerletNeighborList(sigma*2+1, neighbor_skin, box_size)

    trajectories = [positions.copy()] if collect_trajectory else None

    output_handle = None
    output_path = None
    if name:
        output_path = datadir  / name
        output_handle = open(output_path, 'w')
        
    #write_xyz(output_handle, positions, 0, summing_forces,box_size)

    for step in range(num_steps):
        if step % 100 == 0:
            print(f"Step {step}/{num_steps}")

        noise = rng.normal(0, np.sqrt(2 * temperature * gamma / dt), size=(num_particles, 2))
        f_active += (-f_active/tau + noise) * dt


        neighbor_pairs = None
        if neighbor_list is not None:
            neighbor_pairs = neighbor_list.get_pairs(positions)

        lj_force, scalarFLJ = compute_soft_lj_forces(
            positions, box_size, epsilon, sigma, rcut, lambda_param, neighbor_pairs=neighbor_pairs
        )
    
        # if scalarFLJ < min_ljForce:
        #     min_ljForce = scalarFLJ
        if use_harmonic_trap:     
            trap_force = harmonic_force(positions, kappa, center)
        else:
            trap_force = 0


        positions += (trap_force + f_active+ lj_force) * dt / gamma 
        forces_x = f_active[:,0] / gamma # force times mobility matrix (in units of speed)
        forces_y = f_active[:,1] / gamma # force times mobility matrix (in units of speed)
        summing_forces = forces_x**2 + forces_y**2
        #print("Dimensions of summing_forces",summing_forces.shape)

        #
        #alert if we have crossed the walls
        if np.any(positions < -box_size/2) or np.any(positions > box_size/2):
            print("Alert: Particles have crossed the walls")

        positions = (positions + box_size / 2) % box_size - box_size / 2


        if collect_trajectory:
            trajectories.append(positions.copy())

        if output_handle and step % frame == 0 and step > equilibration_steps:
            write_xyz(output_handle, positions, step,summing_forces, box_size)

    if output_handle:
        output_handle.close()
        print("Simulation completed. Output written to " + name)
    else:
        print("Simulation completed.")
    print("Minimum force " + str(min_ljForce)+"for params eps"+str(epsilon)+" sig"+str(sigma)+" rc"+str(rcut))

    if collect_trajectory:
        return np.array(trajectories)
    return None



In [ ]:
## Standard Parameters
folder = "/PersistentHarmonic/"
tau_mem0 = 3
kappa = 1.0
tau0 = 5
gamma0 = 3
seed = 0
A_x=0.5
L_x=5.0
for s in range(1):
    seed+=1
    epsilon = 0.0 #1.0 #0.5 #0.5
    sigma = 1.1 #+ee*0.5
    lambda_param = 0.1 #1.0
    use_harmonic_trap = True #False #False # Set to False to disable the trap
    use_arbitrary_trap = False #True #False #True #False # Set to False to disable the trap
    #kappa = 0.5  # Spring constant for the harmonic trap
    Center = np.array([0,0]) #Center of harmonic trap
    pair_interactions = True # Set to True to enable Lennard-Jones pair interactions
    num_particles = 100
    box_size = 100  # Box is a cube of size box_size x box_size
    temperature = 0.1 #0.01 for easily inferring forces
    #gamma = 1 # Friction coefficient
    dt = 0.001  # Time step
    frame = 100 #100
    #tau = 10
    num_steps = 30000 
    equilibration_steps = 30000 #Steps to equilibrate before printing
    rcut= sigma*2+1 #
    neighbor_skin = 0.5
    name_anchor="output_Test"
    name = name_anchor+".xyz"
    # running the simulation:
    OD_MD_SimulationLJPairs(
        num_particles, box_size, kappa, temperature, gamma0, dt,
        num_steps, equilibration_steps, frame, use_harmonic_trap,
        name, Center, epsilon, sigma, rcut, lambda_param,
        use_neighbor_list=True, neighbor_skin=neighbor_skin,
        random_seed=seed, walls = True,tau = tau0, use_arbitrary_trap=True, kappa_y=kappa, A_x=A_x, L_x=L_x
    )
    df_atoms = parse_lammps_xyz(datadir / name)
    df_save =  df_atoms.rename(columns={'id': 'particle'})
    df_save = df_save.rename(columns={'timestep': 'frame'})
    # re-name frames to be 0,1,2,3...
    df_save['frame'] = df_save['frame'].astype(int)
    # take off min frame number
    df_save['frame'] = df_save['frame'] - df_save['frame'].min()
    # divide by frame rate

    df_save['frame'] = df_save['frame'] / frame
    dataname =  name_anchor+".csv"
    df_save.to_csv(datadir_csv + folder + dataname, index=False)


Step 0/60000
Step 100/60000
Step 200/60000
Step 300/60000
Step 400/60000
Step 500/60000
Step 600/60000
Step 700/60000
Step 800/60000
Step 900/60000
Step 1000/60000
Step 1100/60000
Step 1200/60000
Step 1300/60000
Step 1400/60000
Step 1500/60000
Step 1600/60000
Step 1700/60000
Step 1800/60000
Step 1900/60000
Step 2000/60000
Step 2100/60000
Step 2200/60000
Step 2300/60000
Step 2400/60000
Step 2500/60000
Step 2600/60000
Step 2700/60000
Step 2800/60000
Step 2900/60000
Step 3000/60000
Step 3100/60000
Step 3200/60000
Step 3300/60000
Step 3400/60000
Step 3500/60000
Step 3600/60000
Step 3700/60000
Step 3800/60000
Step 3900/60000
Step 4000/60000
Step 4100/60000
Step 4200/60000
Step 4300/60000
Step 4400/60000
Step 4500/60000
Step 4600/60000
Step 4700/60000
Step 4800/60000
Step 4900/60000
Step 5000/60000
Step 5100/60000
Step 5200/60000
Step 5300/60000
Step 5400/60000
Step 5500/60000
Step 5600/60000
Step 5700/60000
Step 5800/60000
Step 5900/60000
Step 6000/60000
Step 6100/60000
Step 6200/60000
Step